# wrangle / portfolio / allocation

**Intent**
- Estimates the risky share and the risk-adjusted return from `portfolio_allocation_volatility_pct`, and must:
  - Give `portfolio_allocation_risky_pct`, the risky share that maximises utility, capped at 100.
  - Give `portfolio_allocation_riskadjusted_pct`, the risk-adjusted excess return in percent a year.
  - Take both from one excess return and risk aversion, so they cannot disagree.
  - Treat the share as the Merton share, an upper bound.
  - Put a 100 percent equity portfolio and a 60/40 one on one scale by the risk-adjusted return.
  - Hold the rmse of each output on `validation` and `backtest` to 0.01.

**Data**
- Personal data is git-ignored in `data/`, with synthetic placeholders committed.
- `training` is ten volatilities with both outputs, `validation` is five further, and `backtest` is ten across the range.
- `backtest` includes capped shares and is held back for acceptance.
- `golden` is volatilities from 5 to 60 percent, then a request with the input missing.

**Status**
- The excess return and the risk aversion are recovered from synthetic points.

**Future**
- Overall: the work still to do before the first release:
  - Promote the target from local to served, then add the Register stage, once the limits are fixed against real data.
  - Add `query_postgres` to `collect_run` and load the real series in place of the synthetic ones.
  - Replace the fit with the Merton share, `(mu - r) / (gamma * sigma^2)`, from `portfolio/return`, `economy/rates` and `portfolio/utility`.
  - Report the share and the risk-adjusted return at low, central and high risk aversion.
  - Treat the share as an upper bound, since it assumes constant returns, no income floor and no taxes.
  - Get excess return and volatility per asset class, leverage limits and the cash rate history.
  - Fix the limits against real data, then register and review before any release.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("wrangle/portfolio/allocation")


GOLDEN = "portfolio_allocation_golden"

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run


def labelled(volatilities):
    adjusted = [4.0 - 0.5 * 3.0 * (volatility / 100.0) ** 2 * 100.0 for volatility in volatilities]
    shares = [min(100.0, 4.0 / 3.0 / (volatility / 100.0) ** 2) for volatility in volatilities]
    return {ws.inputs[0]: volatilities, ws.outputs[0]: [round(share, 6) for share in shares], ws.outputs[1]: [round(value, 6) for value in adjusted]}


collect_run(
    ws,
    1000,
    label="Ten synthetic volatilities with both outputs",
    name="portfolio_allocation_training",
    source="builder",
    build=lambda: labelled([5.0, 8.0, 10.0, 12.0, 15.0, 16.0, 18.0, 20.0, 22.0, 25.0]),
)
collect_run(ws, 1000, label="Five further synthetic volatilities", name="portfolio_allocation_validation", source="builder", build=lambda: labelled([6.0, 9.0, 14.0, 19.0, 24.0]))
collect_run(
    ws,
    1000,
    label="Ten synthetic volatilities across the range",
    name="portfolio_allocation_backtest",
    source="builder",
    build=lambda: labelled([7.0, 11.0, 13.0, 17.0, 21.0, 23.0, 26.0, 28.0, 30.0, 32.0]),
)

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "portfolio_allocation_training"), validation=ws.run(1000, "portfolio_allocation_validation"), backtest=ws.run(1000, "portfolio_allocation_backtest"))
print("training", ws.training.height, "validation", ws.validation.height, "backtest", ws.backtest.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
import numpy as np

variance = (ws.training[ws.inputs[0]].to_numpy() / 100.0) ** 2 * 100.0
slope, intercept = np.polyfit(variance, ws.training[ws.outputs[1]].to_numpy(), 1)
ws.params = {"excess": round(float(intercept), 4), "gamma": round(float(-2.0 * slope), 4)}
print("fitted", ws.params)
assert ws.params == {"excess": 4.0, "gamma": 3.0}, "the fit did not recover the excess return and risk aversion"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

EXCESS, GAMMA = ws.params["excess"], ws.params["gamma"]


def allocation(volatility):
    share = 100.0 if volatility <= 0.0 else min(100.0, EXCESS / GAMMA / (volatility / 100.0) ** 2)
    return share, EXCESS - 0.5 * GAMMA * (volatility / 100.0) ** 2 * 100.0


define_model(
    ws,
    model=1000,
    flavour="pyfunc",
    source=allocation,
    label="Merton share and risk adjusted return, least squares on the variance",
    constants={"EXCESS": ws.params["excess"], "GAMMA": ws.params["gamma"]},
)

## Evaluate ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-evaluate))

In [ ]:
from mlmodels import evaluate_gate, evaluate_metrics, evaluate_model, evaluate_models

LIMITS = dict.fromkeys(ws.outputs, 0.01)


def acceptance_checks():
    checks = []
    for split in ("validation", "backtest"):
        frame = getattr(ws, split)
        predicted = evaluate_model(ws, 1000, split)
        checks.extend(
            {
                "split": split,
                "gate": split == "backtest",
                "output": output,
                "name": "rmse",
                "value": evaluate_metrics(frame[output], predicted[output])["rmse"],
                "limit": LIMITS[output],
                "direction": "max",
            }
            for output in ws.outputs
        )
    return checks


evaluate_gate(ws, 1000, acceptance_checks)
evaluate_models(ws)

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = {ws.inputs[0]: [5.0, 10.0, 12.2, 15.0, 20.0, 30.0, 60.0, None], "expect": ["ok"] * 7 + ["error"]}
package_model(ws, 1000, golden, target={"kind": "local"})